# DDIM Step — 面试版

**难度：** Medium

**面试要点：** 展开公式推导，解释每一步的物理含义

In [ ]:
import torch

In [ ]:
# ✅ INTERVIEW

def ddim_step(x_t, noise_pred, alpha_bar_t, alpha_bar_prev):
    # ---- 面试推导版 ----
    # 
    # 扩散模型前向加噪: x_t = √ᾱ_t · x_0 + √(1-ᾱ_t) · ε, ε ~ N(0,I)
    # 
    # Step 1: 反推 x_0 的预测值
    #   x_0 = (x_t - √(1-ᾱ_t) · ε) / √ᾱ_t
    sqrt_alpha_t = alpha_bar_t ** 0.5
    sqrt_one_minus_alpha_t = (1 - alpha_bar_t) ** 0.5
    sqrt_alpha_prev = alpha_bar_prev ** 0.5
    sqrt_one_minus_alpha_prev = (1 - alpha_bar_prev) ** 0.5

    x0_pred = (x_t - sqrt_one_minus_alpha_t * noise_pred) / sqrt_alpha_t

    # Step 2: 指向 x_t 方向的"噪声分量"
    # DDIM 核心: 用 √(1-ᾱ_{t-1})·ε 代替随机噪声
    # 这使得采样过程完全确定性
    noise_dir = sqrt_one_minus_alpha_prev * noise_pred

    # Step 3: 合成 x_{t-1}
    # x_{t-1} = √ᾱ_{t-1}·x_0_pred + √(1-ᾱ_{t-1})·ε
    # 这其实就是用 x_0_pred 重新走一遍前向加噪公式，但到 t-1 步
    x_prev = sqrt_alpha_prev * x0_pred + noise_dir
    return x_prev

In [ ]:
torch.manual_seed(42)

T = 5
alpha_bars = torch.linspace(0.99, 0.01, T + 1)

x_clean = torch.tensor([1.0, -1.0, 0.5])
noise   = torch.randn_like(x_clean)

ab_T = alpha_bars[T]
x_t  = ab_T ** 0.5 * x_clean + (1 - ab_T) ** 0.5 * noise

for step in range(T, 0, -1):
    ab_t    = alpha_bars[step]
    ab_prev = alpha_bars[step - 1]
    noise_pred = (x_t - ab_t ** 0.5 * x_clean) / (1 - ab_t) ** 0.5
    x_t = ddim_step(x_t, noise_pred, ab_t, ab_prev)

print(f"Final: {x_t.tolist()}")
print(f"Clean: {x_clean.tolist()}")

In [ ]:
from torch_judge import check
check("ddim_step")

## 📝 核心思路总结

1. **变量命名**：`sqrt_alpha_t` 等中间变量让公式更清晰
2. **物理含义**：预测 x0 → 重新加噪到 t-1 步
3. **确定性采样**：DDIM 的关键优势，可复现结果